# Detección de Elementos de Protección Personal (PPE) con Roboflow y YOLO V8 nano.
## **Por**: Juan Guillermo Escobar Baez

Este cuaderno describe el proceso para importar un dataset etiquetado desde Roboflow para, posteriormente, entrenarlo mediante transfer learning, utilizando YOLO (You Only Look Once) v8. Se utilizó un dataset de 730 imágenes, de las cuales 2 fueron recopiladas manualmente y el resto se combinó a partir de 2 datasets (11 de uno y las restantes del otro).

* Dado que no se cuenta con ultralytics ni roboflow instalados por defecto, es necesario instalarlos en el environment.

In [ ]:
!pip install roboflow ultralytics



## 1) Descargar dataset desde Roboflow
# Se descarga la version 1 del proyecto `epp3` en formato YOLOv8.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="1**********")
project = rf.workspace("juans-workspace-agc9e").project("epp3")
version = project.version(1)
dataset = version.download("yolov8")


# %% [markdown]
# ## 2) Cargar modelo base y entrenar
# Se usa `yolov8n.pt` como punto de partida y `./epp3-1/data.yaml` como configuracion del dataset.


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to epp3-1 in yolov8:: 100%|██████████| 1461/1461 [00:00<00:00, 3404.95it/s]


loading Roboflow workspace...
loading Roboflow project...


## 2) Importar el modelo v8 nano de YOLO
# Se importa la versión nano de YOLO 8, mediante yolov8n.pt

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

## 1) Entrenar con YOLO
# Se descarga la version 1 del proyecto `epp3` en formato YOLOv8. Además, se configuran 100 épocas, un tamaño de imagen de 640x640 y una paciencia de 50

In [ ]:
path = "./epp3-1/data.yaml"

results = model.train(data=path, epochs=100, imgsz=640, patience=50)


New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.39 🚀 Python-3.10.20 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./epp3-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_sc

* Es posible concluir que el modelo es bueno para ser un prototipo rápido (entrenado rápidamente), aunque no detecta bien varias clases; se entrenó con pocas imágenes.

* Finalmente, el mejor modelo se almacena en una carpeta derivada del entrenamiento, donde se puede encontrar como best.pt. A partir de este se realizó una detección de objetos con una imagen de prueba que el modelo no había visto antes.

In [ ]:
model = YOLO('runs/detect/train-3/weights/best.pt')

In [ ]:
res = model("./proof2.jpg", imgsz=640)

# Muestra el resultado en el notebook
import matplotlib.pyplot as plt

res_ploteado = res[0].plot() # Dibuja las cajas y etiquetas
plt.imshow(res_ploteado)
plt.axis('off')

# %% [markdown]
# ## 4) Verificacion rapida del entorno



image 1/1 /home/jge25/epp2/proof2.jpg: 448x640 1 NO-Mask, 1 NO-Safety Vest, 1 helmet, 1 worker, 54.5ms
Speed: 4.1ms preprocess, 54.5ms inference, 2.2ms postprocess per image at shape (1, 3, 448, 640)


(np.float64(-0.5), np.float64(1469.5), np.float64(979.5), np.float64(-0.5))

* Se observa la versión de Python instalada en el environment.

In [ ]:
!python --version

Python 3.10.20
